# RAG: Baseline Bi-Encoder Retrieval vs. Cross-Encoder Re-Ranking

**Document:** EU General Data Protection Regulation (GDPR, Regulation 2016/679), the full official text from EUR-Lex (`data/gdpr.pdf`).

**Pipelines** (both run over the *same* chunk corpus):
- **Pipeline 1, Baseline:** bi-encoder (`all-MiniLM-L6-v2`) → FAISS top 10 → take top 3 → LLM produces **Answer 1**
- **Pipeline 2, Re-ranking:** same top 10 → cross-encoder (`ms-marco-MiniLM-L-6-v2`) scores each (query, chunk) pair → re-sort → top 3 → LLM (identical prompt) produces **Answer 2**

Notebook flow: setup → load and chunk → index → pipeline functions → failure-query hunt → rank-shift tables → answer comparison → latency profiling → scaling experiment.

## 1. Setup and configuration

In [1]:
import os, re, time, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf                   # PDF text extraction
import faiss                     # vector index
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from dotenv import load_dotenv

load_dotenv(Path("../.env"))     # reads GEMINI_API_KEY from the project-root .env if one exists

PDF_PATH = Path("data/gdpr.pdf")
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Chunking: about 220 words is roughly 300 word-piece tokens, safely under the cross-encoder's 512-token limit.
CHUNK_WORDS   = 220
OVERLAP_WORDS = 40

TOP_K_RETRIEVE = 10   # candidates from the bi-encoder
TOP_K_FINAL    = 3    # chunks passed to the LLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 140)
print("Device:", DEVICE)

C:\Users\Shubh\Desktop\LMA\Rag-assignment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## 2. Load the PDF and chunk it

The Official Journal repeats a running header and footer on each page (date, "Official Journal of the European Union", page code such as `L 119/1`). These lines are noise for retrieval, so they are removed before chunking.

Chunking uses fixed-size word windows with overlap. Each chunk records the page where it starts, so answers can be traced back to the PDF.

In [2]:
# Running header/footer lines printed on each page of the Official Journal.
NOISE_LINE = re.compile(
    r"^\s*(\d{1,2}\.\d{1,2}\.\d{4}|EN|Official Journal of the European Union|L\s?\d+/\d+)\s*$"
)

def load_pages(path):
    """Returns a list of (page_number, cleaned_text) pairs."""
    pages = []
    with pymupdf.open(path) as doc:
        for i, page in enumerate(doc, start=1):
            lines = [l for l in page.get_text().splitlines() if not NOISE_LINE.match(l)]
            text = " ".join(lines)
            text = re.sub(r"(\w)-\s+(\w)", r"\1\2", text)   # rejoin words hyphenated across a line break
            text = re.sub(r"\s+", " ", text).strip()
            pages.append((i, text))
    return pages

def chunk_pages(pages, size=CHUNK_WORDS, overlap=OVERLAP_WORDS):
    """Sliding word-window chunker over the whole document; each word keeps its page number."""
    words = [(w, p) for p, t in pages for w in t.split()]
    chunks, step = [], size - overlap
    for start in range(0, len(words), step):
        window = words[start:start + size]
        if len(window) < 30:          # skip a tiny trailing fragment
            break
        chunks.append({
            "chunk_id": len(chunks),
            "page": window[0][1],
            "text": " ".join(w for w, _ in window),
        })
    return chunks

pages  = load_pages(PDF_PATH)
chunks = chunk_pages(pages)
chunk_texts = [c["text"] for c in chunks]

print(f"Pages: {len(pages)} | Chunks: {len(chunks)}")
print("\nSample chunk:\n", textwrap.fill(chunks[200]["text"][:600], 110))

Pages: 88 | Chunks: 305

Sample chunk:
 natural person; (b) processing on a large scale of special categories of data referred to in Article 9(1), or
of personal data relating to criminal convictions and offences referred to in Article 10; or (c) a systematic
monitoring of a publicly accessible area on a large scale. 4. The supervisory authority shall establish and
make public a list of the kind of processing operations which are subject to the requirement for a data
protection impact assessment pursuant to paragraph 1. The supervisory authority shall communicate those lists
to the Board referred to in Article 68. 5. The supervisory


## 3. Index the chunks with the bi-encoder

Every chunk is embedded **once, offline**. Vectors are L2-normalised, so the inner product in `IndexFlatIP` equals cosine similarity. A flat (exact) index is enough at this corpus size.

In [3]:
bi_encoder    = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device=DEVICE)

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(chunk_texts, batch_size=64, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=True).astype("float32")
index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)
print(f"Indexed {index.ntotal} chunks (dim={chunk_emb.shape[1]}) in {time.perf_counter()-t0:.2f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18151.06it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 15934.08it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

...

Batches:  60%|██████    | 3/5 [00:01<00:00,  2.05it/s]

Batches:  80%|████████  | 4/5 [00:01<00:00,  2.17it/s]

Batches: 100%|██████████| 5/5 [00:02<00:00,  2.42it/s]

Batches: 100%|██████████| 5/5 [00:02<00:00,  2.17it/s]

Indexed 305 chunks (dim=384) in 2.32s


## 4. LLM setup

The default backend is **Gemini** (set `GEMINI_API_KEY` in a `.env` file). Without a key, the notebook falls back to a small local Hugging Face model, so it still runs end to end.

Both pipelines use **the same prompt template** and temperature 0. Only the 3 context chunks differ.

In [4]:
PROMPT_TEMPLATE = """You are answering questions about the EU General Data Protection Regulation (GDPR).
Use ONLY the context passages below. If the context does not contain the answer, say so explicitly.
Quote or cite the relevant Article where possible. Mention any exceptions or conditions that apply.

Context:
{context}

Question: {question}

Answer:"""

def build_prompt(question, ctx_chunks):
    context = "\n\n".join(f"[Passage {i+1} | chunk {c['chunk_id']} | page {c['page']}]\n{c['text']}"
                          for i, c in enumerate(ctx_chunks))
    return PROMPT_TEMPLATE.format(context=context, question=question)

GEMINI_KEY   = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.7-flash")
LOCAL_MODEL  = "Qwen/Qwen2.5-1.5B-Instruct"

if GEMINI_KEY:
    from google import genai
    from google.genai import types
    _client = genai.Client(api_key=GEMINI_KEY)
    LLM_NAME = GEMINI_MODEL
    def _call_llm(prompt):
        r = _client.models.generate_content(
            model=GEMINI_MODEL, contents=prompt,
            config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=2048))
        return (r.text or "[empty response]").strip()
else:
    from transformers import pipeline
    _gen = pipeline("text-generation", model=LOCAL_MODEL, device_map=DEVICE, torch_dtype="auto")
    LLM_NAME = LOCAL_MODEL
    def _call_llm(prompt):
        out = _gen([{"role": "user", "content": prompt}], max_new_tokens=350, do_sample=False)
        return out[0]["generated_text"][-1]["content"].strip()

LLM_RETRIES = 6
LLM_MIN_INTERVAL_S = 13   # pacing to stay under the free tier's 5 requests/minute
LAST_LLM_MS = 0.0         # duration of the most recent *successful* LLM call (excludes retry and pacing waits)
_last_call = 0.0

def llm(prompt):
    """Calls the LLM, retrying on temporary errors (429 rate limit, 5xx overload) with exponential backoff."""
    global LAST_LLM_MS, _last_call
    for attempt in range(LLM_RETRIES):
        time.sleep(max(0.0, _last_call + LLM_MIN_INTERVAL_S - time.time()))
        _last_call = time.time()
        t0 = time.perf_counter()
        try:
            out = _call_llm(prompt)
            LAST_LLM_MS = (time.perf_counter() - t0) * 1000
            return out
        except Exception as e:
            code = getattr(e, "code", None)
            if code not in (429, 500, 503) or attempt == LLM_RETRIES - 1:
                raise
            wait = 5 * 2 ** attempt
            print(f"  LLM unavailable ({code}), retrying in {wait}s...")
            time.sleep(wait)

print("LLM backend:", LLM_NAME)

LLM backend: gemini-3.7-flash


## 5. The two pipelines

- `retrieve_top10`: stage 1, shared by both pipelines
- `rerank`: stage 2, the cross-encoder jointly encodes `[CLS] query [SEP] chunk [SEP]` for each of the 10 candidates
- Each pipeline returns its answer, the top-3 chunk IDs and per-stage timings

In [5]:
def retrieve_top10(query, k=TOP_K_RETRIEVE):
    """Stage 1: bi-encoder dense retrieval. Returns [(chunk_id, cosine_score)], best first."""
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q, k)
    return list(zip(ids[0].tolist(), scores[0].tolist()))

def rerank(query, candidates):
    """Stage 2: cross-encoder scores each (query, chunk) pair; candidates are re-sorted descending."""
    pairs = [(query, chunk_texts[cid]) for cid, _ in candidates]
    ce_scores = cross_encoder.predict(pairs, batch_size=len(pairs))
    order = np.argsort(-ce_scores)
    return [(candidates[i][0], float(ce_scores[i])) for i in order]

def _timed(fn, *args):
    t0 = time.perf_counter(); out = fn(*args)
    return out, (time.perf_counter() - t0) * 1000   # milliseconds

def pipeline1(query, generate=True):
    cands, t_ret = _timed(retrieve_top10, query)
    top3 = [cid for cid, _ in cands[:TOP_K_FINAL]]
    answer = llm(build_prompt(query, [chunks[i] for i in top3])) if generate else None
    t_llm = LAST_LLM_MS if generate else 0.0
    return {"candidates": cands, "top3": top3, "answer": answer,
            "t_retrieve_ms": t_ret, "t_llm_ms": t_llm}

def pipeline2(query, candidates=None, generate=True):
    # Reuses the same top-10 from stage 1, so both pipelines see the same candidate pool.
    t_ret = 0.0
    if candidates is None:
        candidates, t_ret = _timed(retrieve_top10, query)
    reranked, t_rr = _timed(rerank, query, candidates)
    top3 = [cid for cid, _ in reranked[:TOP_K_FINAL]]
    answer = llm(build_prompt(query, [chunks[i] for i in top3])) if generate else None
    t_llm = LAST_LLM_MS if generate else 0.0
    return {"reranked": reranked, "top3": top3, "answer": answer,
            "t_retrieve_ms": t_ret, "t_rerank_ms": t_rr, "t_llm_ms": t_llm}

# Quick check on a simple query.
r1 = pipeline1("What is the right to data portability?", generate=False)
r2 = pipeline2("What is the right to data portability?", r1["candidates"], generate=False)
print("Baseline top3:", r1["top3"], "| Re-ranked top3:", r2["top3"])

Baseline top3: [175, 147, 49] | Re-ranked top3: [176, 175, 53]


## 6. Failure-query hunt

Each candidate query is labelled with **gold evidence**: one or more short text snippets copied verbatim from the article that actually answers it. Any chunk containing a snippet counts as a *gold chunk*. Labelling by snippet rather than by chunk ID keeps the labels valid if chunk sizes change.

For each query the hunt records:
- the gold chunk's rank under the bi-encoder (1–10, or `>10`)
- its rank after cross-encoder re-ranking
- `restored = True` when the baseline missed it in the top 3 **and** re-ranking moved it into the top 3

**Edit this list.** Read the relevant articles, add your own queries, and keep the ones that show the behaviour you want to discuss.

In [6]:
# category, query, gold snippets (lower-case substrings of the true answer's text)
CANDIDATE_QUERIES = [
    # Keyword trap: "staff" and "records" overlap with many passages about records and employees,
    # but the exemption is phrased as "fewer than 250 persons" (Art. 30(5)).
    ("keyword_trap", "Does a startup with 10 staff have to maintain records of processing?",
        ["fewer than 250 persons"]),
    ("keyword_trap", "How long can personal data be stored?",            # answer says "no longer than is necessary"
        ["for no longer than is necessary for the purposes"]),

    # Exception: the general rule exists, but the answer depends on an "except when" clause elsewhere.
    ("exception", "Must every company appoint a data protection officer?",                  # Art. 37(1) conditions
        ["core activities of the controller or the processor consist of processing operations"]),
    ("exception", "Can data collected for one purpose be used for scientific research later?",   # Art. 5(1)(b)
        ["shall, in accordance with article 89(1), not be considered to be incompatible"]),
    ("exception", "Does a controller always have to notify the affected individuals about a data breach?",  # Art. 34(3)
        ["shall not be required if any of the following conditions are met"]),
    ("exception", "Can a company refuse a person's request to have their data erased?",     # Art. 17(3)
        ["paragraphs 1 and 2 shall not apply to the extent that processing is necessary"]),

    # Negation and role constraints
    ("negation", "Is processing someone's religious beliefs or health information prohibited?",  # Art. 9(1)
        ["shall be prohibited"]),
    ("negation", "Can the data protection officer be fired for doing their job?",           # Art. 38(3)
        ["shall not be dismissed or penalised"]),
    ("negation", "In which cases does the right to data portability not apply?",            # Art. 20(3)
        ["that right shall not apply to processing necessary for the performance of a task"]),

    # Buried facts: specific numbers or thresholds
    ("buried", "What is the maximum fine for breaching the basic principles of processing?",     # Art. 83(5)
        ["20 000 000 eur"]),
    ("buried", "At what age can a child give consent for an online service on their own?",  # Art. 8(1)
        ["at least 16 years", "not below 13 years"]),

    # Control: both pipelines retrieve a correct passage (Art. 82 or Recital 146).
    ("control", "Who pays if a person suffers harm because a company mishandled their data?",    # Art. 82
        ["right to receive compensation", "should compensate any damage"]),

    # Degraded case: re-ranking makes things worse. The cross-encoder prefers passages sharing
    # "request", "supervisory authority" and "Member States" (Arts. 61-62, mutual assistance and joint operations)
    # over Art. 64(2), which actually names the Commission.
    ("degraded", "Can the Commission ask the Board to examine a matter that affects more than one Member State?",
        ["the chair of the board or the commission may request that any matter of general application"]),
]

def norm(s):
    """Lower-case, straighten curly quotes (the PDF uses ‘ ’), collapse whitespace."""
    s = s.lower().translate(str.maketrans("‘’“”", "''\"\""))
    return re.sub(r"\s+", " ", s)

norm_chunks = [norm(t) for t in chunk_texts]

def gold_ids(snippets):
    return {i for i, t in enumerate(norm_chunks) if any(norm(s) in t for s in snippets)}

def best_rank(ranked_ids, gold):
    """1-based rank of the highest-placed gold chunk, or None if no gold chunk is in the list."""
    for r, cid in enumerate(ranked_ids, start=1):
        if cid in gold:
            return r
    return None

rows, cache = [], {}
for cat, q, snippets in CANDIDATE_QUERIES:
    gold = gold_ids(snippets)
    if not gold:
        print(f"WARNING: no chunk contains the gold snippet for: {q!r}. Check the snippet text.")
    p1 = pipeline1(q, generate=False)
    p2 = pipeline2(q, p1["candidates"], generate=False)
    cache[q] = (p1, p2, gold)
    b = best_rank([c for c, _ in p1["candidates"]], gold)
    r = best_rank([c for c, _ in p2["reranked"]], gold)
    rows.append({"category": cat, "query": q, "gold_chunks": sorted(gold),
                 "baseline_rank": b, "reranked_rank": r,
                 "restored": (b is None or b > 3) and (r is not None and r <= 3),
                 "degraded": (b is not None and b <= 3) and (r is None or r > 3)})

hunt = pd.DataFrame(rows)
hunt.to_csv(RESULTS_DIR / "failure_hunt.csv", index=False)
hunt

,category,query,gold_chunks,baseline_rank,reranked_rank,restored,degraded
0,keyword_trap,Does a startup with 10 staff have to maintain records of processing?,[193],7.0,3.0,True,False
1,keyword_trap,How long can personal data be stored?,[146],NaN,NaN,False,False
2,exception,Must every company appoint a data protection officer?,[205],4.0,1.0,True,False
3,exception,Can data collected for one purpose be used for scientific research later?,[146],9.0,2.0,True,False
4,exception,Does a controller always have to notify the affected individuals about a data breach?,"[197, 198]",4.0,2.0,True,False
5,exception,Can a company refuse a person's request to have their data erased?,"[172, 173]",3.0,3.0,False,False
6,negation,Is processing someone's religious beliefs or health information prohibited?,[153],10.0,2.0,True,False
7,negation,Can the data protection officer be fired for doing their job?,[207],1.0,1.0,False,False
8,negation,In which cases does the right to data portability not apply?,[176],3.0,1.0,False,False
9,buried,What is the maximum fine for breaching the basic principles of processing?,"[291, 292]",7.0,1.0,True,False


## 7. Rank-shift analysis for the selected failure queries

For each chosen query, the table shows all 10 candidates with their bi-encoder rank and score and their cross-encoder rank and score. Gold chunks are marked with ★.

In [7]:
# By default, every query that re-ranking restored, plus any it degraded. You can also choose by hand:
# SELECTED = ["...", "..."]
SELECTED = hunt.loc[hunt.restored | hunt.degraded, "query"].tolist()
print(f"{len(SELECTED)} queries selected")

def rank_shift_table(q):
    p1, p2, gold = cache[q]
    ce_rank  = {cid: r for r, (cid, _) in enumerate(p2["reranked"], start=1)}
    ce_score = dict(p2["reranked"])
    rows = []
    for r, (cid, s) in enumerate(p1["candidates"], start=1):
        rows.append({"chunk": cid, "gold": "★" if cid in gold else "", "page": chunks[cid]["page"],
                     "bi_rank": r, "bi_score": round(s, 3),
                     "ce_rank": ce_rank[cid], "ce_score": round(ce_score[cid], 3),
                     "shift": f"#{r} → #{ce_rank[cid]}",
                     "preview": chunk_texts[cid][:110] + "…"})
    return pd.DataFrame(rows).sort_values("ce_rank")

for q in SELECTED:
    print("\n" + "=" * 110 + f"\nQUERY: {q}")
    display(rank_shift_table(q))

7 queries selected

QUERY: Does a startup with 10 staff have to maintain records of processing?


,chunk,gold,page,bi_rank,bi_score,ce_rank,ce_score,shift,preview
1,67,,16,2,0.425,1,-6.222,#2 → #1,"the processor to the controller, setting out the subjectmatter and duration of the processing, the nature and …"
8,8,,3,9,0.385,2,-7.262,#9 → #2,States as well as effective cooperation between the supervisory authorities of different Member States. The pr…
6,193,★,51,7,0.391,3,-9.076,#7 → #3,"data to a third country or an international organisation, including the identifi­ cation of that third country…"
5,28,,7,6,0.396,4,-9.715,#6 → #4,"for which they are processed. This requires, in particular, ensuring that the period for which the personal da…"
4,200,,53,5,0.397,5,-10.313,#5 → #5,"natural person; (b) processing on a large scale of special categories of data referred to in Article 9(1), or …"
9,147,,36,10,0.384,6,-10.446,#10 → #6,no longer than is necessary for the purposes for which the personal data are processed; personal data may be s…
3,124,,29,4,0.398,7,-10.467,#4 → #7,"context, in particular for the conditions under which personal data in the employment context may be processed…"
7,195,,52,8,0.389,8,-10.630,#8 → #8,the processing. 2. In assessing the appropriate level of security account shall be taken in particular of the …
2,188,,49,3,0.401,9,-10.795,#3 → #9,the conditions referred to in paragraphs 2 and 4 for engaging another processor; (e) taking into account the n…
0,296,,84,1,0.427,10,-11.048,#1 → #10,"in particular for the purposes of the recruitment, the performance of the contract of employment, including di…"



QUERY: Must every company appoint a data protection officer?


,chunk,gold,page,bi_rank,bi_score,ce_rank,ce_score,shift,preview
3,205,★,55,4,0.592,1,1.370,#4 → #1,"processing by a controller for the performance of a task carried out by the controller in the public interest,…"
0,206,,55,1,0.688,2,1.351,#1 → #2,"establishment. 3. Where the controller or the processor is a public authority or body, a single data protectio…"
2,208,,56,3,0.622,3,1.274,#3 → #3,"officer shall be bound by secrecy or confidentiality concerning the performance of his or her tasks, in accord…"
1,207,,55,2,0.668,4,0.711,#2 → #4,Article 38 Position of the data protection officer 1. The controller and the processor shall ensure that the d…
5,108,,25,6,0.569,5,-0.842,#6 → #5,of Individuals with Regard to the Processing of Personal Data established by Directive 95/46/EC. It should con…
6,85,,20,7,0.556,6,-2.153,#7 → #6,no longer ensures an adequate level of data protection. Consequently the transfer of personal data to that thi…
8,231,,63,9,0.549,7,-3.277,#9 → #7,"of personal data, measures to ensure data security, and the requirements in respect of onward transfers to bod…"
4,272,,75,5,0.578,8,-4.046,#5 → #8,be adopted within two weeks by simple majority of the members of the Board. Article 67 Exchange of information…
7,86,,20,8,0.555,9,-5.184,#8 → #9,"the rights of the data subjects appropriate to processing within the Union, including the availability of enfo…"
9,247,,68,10,0.542,10,-5.372,#10 → #10,"inform the complainant of the progress and the outcome of the investigation within a reasonable period, in par…"



QUERY: Can data collected for one purpose be used for scientific research later?


...

,chunk,gold,page,bi_rank,bi_score,ce_rank,ce_score,shift,preview
7,61,,15,8,0.468,1,-0.067,#8 → #1,"natural persons, of varying likelihood and severity, may result from personal data processing which could lead…"
9,153,★,37,10,0.467,2,-0.198,#10 → #2,and to the extent that consent is given or authorised by the holder of parental responsibility over the child.…
1,44,,11,2,0.500,3,-1.573,#2 → #3,result in personal data being processed for other purposes by third parties such as employers or insurance and…
2,40,,10,3,0.496,4,-1.864,#3 → #4,for compliance with a legal obligation or for the performance of a task carried out in the public interest or …
5,154,,38,6,0.470,5,-2.421,#6 → #5,(a) the data subject has given explicit consent to the processing of those personal data for one or more speci…
8,156,,38,9,0.467,6,-4.193,#9 → #6,health or social care systems and services on the basis of Union or Member State law or pursuant to contract w…
3,49,,12,4,0.492,7,-4.261,#4 → #7,the age of the data and any appropriate safeguards adopted should be taken into consideration. (63) A data sub…
4,173,,44,5,0.477,8,-5.887,#5 → #8,"by such controllers of any links to, or copy or replication of, those personal data. 3. Paragraphs 1 and 2 sha…"
0,299,,85,1,0.514,9,-7.136,#1 → #9,"Member State law or rules established by national competent bodies, to an obligation of professional secrecy o…"
6,151,,37,7,0.468,10,-7.595,#7 → #10,"processing; (b) the context in which the personal data have been collected, in particular regarding the relati…"



QUERY: What is the maximum fine for breaching the basic principles of processing?


,chunk,gold,page,bi_rank,bi_score,ce_rank,ce_score,shift,preview
6,291,★,82,7,0.523,1,1.505,#7 → #1,"the processor pursuant to Articles 8, 11, 25 to 39 and 42 and 43; (b) the obligations of the certification bod…"
2,289,,82,3,0.560,2,-3.568,#3 → #2,whether to impose an administrative fine and deciding on the amount of the administrative fine in each individ…
0,290,,82,1,0.623,3,-3.835,#1 → #3,"to what extent, the controller or processor notified the infringement; (i) where measures referred to in Artic…"
8,118,,27,9,0.508,4,-5.875,#9 → #4,"obtained through infringements of this Regulation. However, the imposition of criminal penalties for infringem…"
7,116,,27,8,0.521,5,-6.067,#8 → #5,"entire damage. However, where they are joined to the same judicial proceedings, in accordance with Member Stat…"
1,117,,27,2,0.567,6,-6.144,#2 → #6,of a minor infringement or if the fine likely to be imposed would constitute a disproportionate burden to a na…
5,288,,81,6,0.535,7,-6.288,#6 → #7,"the data subject. 5. Where a controller or processor has, in accordance with paragraph 4, paid full compensati…"
9,292,★,83,10,0.503,8,-7.733,#10 → #8,provide access in violation of Article 58(1). 6. Non-compliance with an order by the supervisory authority as …
4,293,,83,5,0.543,9,-8.544,#5 → #9,"imposed by competent national courts, while ensuring that those legal remedies are effective and have an equiv…"
3,120,,28,4,0.553,10,-9.301,#4 → #10,fine is imposed by competent national courts as a criminal penalty and in Estonia the fine is imposed by the s…



QUERY: Can the Commission ask the Board to examine a matter that affects more than one Member State?


,chunk,gold,page,bi_rank,bi_score,ce_rank,ce_score,shift,preview
5,261,,73,6,0.495,1,-1.095,#6 → #1,processor has establishments in several Member States or where a significant number of data subjects in more t…
6,260,,72,7,0.491,2,-1.546,#7 → #2,"Article within one month of receiving the request of another supervisory authority, the requesting supervisory…"
0,273,,76,1,0.557,3,-3.158,#1 → #3,in accordance with that Member State's law. 5. The Commission shall have the right to participate in the activ…
7,106,,25,8,0.489,4,-4.119,#8 → #4,also apply where any supervisory authority concerned or the Commission requests that such matter should be han…
8,265,,74,9,0.483,5,-4.369,#9 → #5,"Board with a view to obtaining an opinion, in particular where a competent supervisory authority does not comp…"
4,112,,26,5,0.496,6,-5.211,#5 → #6,corrective and authorisation powers by the supervisory authority or the dismissal or rejection of complaints. …
3,268,,75,4,0.507,7,-5.229,#4 → #7,for the main establishment; (c) where a competent supervisory authority does not request the opinion of the Bo…
1,264,★,73,2,0.515,8,-7.433,#2 → #8,"to adopt any of the measures below. To that end, the competent supervisory authority shall communicate the dra…"
9,213,,57,10,0.480,9,-7.532,#10 → #9,"in the situation referred to in paragraph 3 of this Article, provides appropriate safeguards. 8. Where the opi…"
2,113,,26,3,0.515,10,-9.050,#3 → #10,"ruling on the interpretation of Union law, including this Regulation. Furthermore, where a decision of a super…"


## 8. Answer 1 vs. Answer 2

Generates both answers for each selected query. Things to check: did Answer 1 **hallucinate**, **leave out** an exception or condition, or **misstate** the rule because the gold chunk was outside its top 3?

In [8]:
answer_rows = []
for q in SELECTED:
    p1_cached, _, gold = cache[q]
    p1 = pipeline1(q)                                   # generates Answer 1
    p2 = pipeline2(q, p1_cached["candidates"])          # same top 10, re-ranked, generates Answer 2
    answer_rows.append({"query": q,
                        "baseline_top3": p1["top3"], "reranked_top3": p2["top3"],
                        "gold_in_baseline_top3": bool(gold & set(p1["top3"])),
                        "gold_in_reranked_top3": bool(gold & set(p2["top3"])),
                        "answer_1_baseline": p1["answer"], "answer_2_reranked": p2["answer"],
                        "t_llm_1_ms": p1["t_llm_ms"], "t_llm_2_ms": p2["t_llm_ms"]})
    print("\n" + "=" * 110)
    print("QUERY:", q)
    print(f"\n--- Answer 1 (baseline, top3={p1['top3']}) ---\n{textwrap.fill(p1['answer'], 110)}")
    print(f"\n--- Answer 2 (re-ranked, top3={p2['top3']}) ---\n{textwrap.fill(p2['answer'], 110)}")

pd.DataFrame(answer_rows).to_csv(RESULTS_DIR / "answers.csv", index=False)


QUERY: Does a startup with 10 staff have to maintain records of processing?

--- Answer 1 (baseline, top3=[296, 67, 188]) ---
Based on the provided context, it is not possible to answer whether a startup with 10 staff specifically has
to maintain records of processing activities, as the passages do not contain information or exemptions
regarding organization size or employee thresholds.   The context only states generally in Recital (82)
(Passage 2) that: > *"In order to demonstrate compliance with this Regulation, the controller or processor
should maintain records of processing activities under its responsibility. Each controller and processor
should be obliged to cooperate with the supervisory authority and make those records, on request, available to
it, so that it might serve for monitoring those processing operations."*  Because the specific rules,
conditions, or exceptions relating to staff size (such as Article 30(5)) are not included in the provided
text, the context does not


QUERY: Must every company appoint a data protection officer?

--- Answer 1 (baseline, top3=[206, 207, 208]) ---
Based on the provided context, **no, not every company is required to appoint a data protection officer.**
According to Passage 1 (paragraph 4): * In cases other than those referred to in paragraph 1, a controller or
processor **"may or, where required by Union or Member State law shall, designate a data protection
officer."**   **Conditions and Missing Context:** * Designation is voluntary ("may") unless required by Union
or Member State law ("shall"). * While paragraph 4 refers to mandatory conditions in "paragraph 1", the
context does not contain the text of paragraph 1, so the full set of specific cases where a company is
strictly mandated to appoint a DPO is not explicitly detailed in the provided text.

--- Answer 2 (re-ranked, top3=[205, 206, 208]) ---
Based on the provided context, **no**, not every company is required to appoint a data protection officer
(DPO).   Un


QUERY: Can data collected for one purpose be used for scientific research later?

--- Answer 1 (baseline, top3=[125, 48, 127]) ---
Based on the provided context, **yes**, personal data collected for one purpose can be further processed for
another purpose, such as scientific research, provided specific rules, safeguards, and conditions are met.
*(Note: The provided passages cite Recitals rather than specific GDPR Articles; citations below reference
these passages and recitals).*  ### Conditions and Requirements * **Information requirements:** Generally,
where a controller intends to process personal data for a purpose other than that for which they were
collected, the controller must provide the data subject prior to that further processing with information on
that other purpose and other necessary information (Passage 2 / Recital 61–62). * **Safeguards and data
minimization:** Processing for scientific research purposes is to be carried out when the controller has
assessed the feasib


QUERY: Does a controller always have to notify the affected individuals about a data breach?

--- Answer 1 (baseline, top3=[70, 196, 69]) ---
Based on the provided context, **no**, a controller does not always have to notify affected individuals (data
subjects) about a personal data breach.   According to **Recital (86)** (Passage 1), notification is subject
to specific conditions:  * **Condition for notification:** The controller is required to communicate the
breach to the data subject only **"where that personal data breach is likely to result in a high risk to the
rights and freedoms of the natural person"** in order to allow them to take necessary precautions.  * **Timing
and cooperation:** When required, the communication must be made "without undue delay," "as soon as reasonably
feasible," and "in close cooperation with the supervisory authority." * **Content:** The notification must
describe the nature of the personal data breach and provide recommendations for the individual 


QUERY: Is processing someone's religious beliefs or health information prohibited?

--- Answer 1 (baseline, top3=[299, 44, 40]) ---
Based on the provided context, the passages do not explicitly define or list "religious beliefs" or "health
information" as prohibited categories. Therefore, the context does not explicitly state that processing these
specific items is prohibited.   However, the context does reference a **"general prohibition for processing
such special categories of personal data"** [Passage 3, Recital (51)–(52)] and outlines several derogations,
exceptions, and conditions that relate to health and religious bodies:  ### Exceptions and Conditions
Regarding Health According to Passage 3 [Recital (52)], derogations from the prohibition on processing special
categories of personal data are allowed: *   When provided for in Union or Member State law and subject to
suitable safeguards. *   Where it is in the public interest, specifically:     *   *"in particular processing
pe


QUERY: What is the maximum fine for breaching the basic principles of processing?

--- Answer 1 (baseline, top3=[290, 117, 289]) ---
Based on the provided context, it is not explicitly stated what the maximum fine is for breaching the basic
principles of processing.   While Passage 1 provides administrative fine tiers up to "10 000 000 EUR, or in
the case of an undertaking, up to 2 % of the total worldwide annual turnover of the preceding financial year,
whichever is higher" for certain obligations (such as Articles 8, 11, 25 to 39, 42, and 43), paragraph 5 cuts
off and does not contain the provisions or fine amounts related to the basic principles of processing.

--- Answer 2 (re-ranked, top3=[291, 289, 290]) ---
Based on the provided context, infringements of **the basic principles for processing** (including conditions
for consent, pursuant to **Articles 5, 6, 7, and 9**) are subject to administrative fines of:  * Up to **20
000 000 EUR**, or  * In the case of an undertaking, up to


QUERY: Can the Commission ask the Board to examine a matter that affects more than one Member State?

--- Answer 1 (baseline, top3=[273, 264, 113]) ---
Yes. According to the provided context, the Commission can request that such a matter be examined by the
Board.  Specifically, Passage 2 (paragraph 2) states: > *"Any supervisory authority, the Chair of the Board or
the Commission may request that any matter of general application or producing effects in more than one Member
State be examined by the Board with a view to obtaining an opinion[...]"*  **Conditions and particular
applications mentioned:** * The examination is requested **"with a view to obtaining an opinion."** * The
passage highlights that this applies **"in particular where a competent supervisory authority does not comply
with the obligations for mutual assistance in accordance with Article 61 or for joint operations in accordance
with Article 62."**   *(Note: While the specific Article heading for this paragraph is not

## 9. Latency profiling

Each stage is timed separately with `time.perf_counter()`, after a warm-up run, since the first call includes model and CUDA initialisation.
- **Retrieval:** query embedding plus FAISS search (repeated `N_RUNS` times per query)
- **Re-rank:** cross-encoder over 10 pairs (repeated `N_RUNS` times per query)
- **LLM:** measured once per query in section 8, to save API quota

In [9]:
N_RUNS = 5
queries = [q for _, q, _ in CANDIDATE_QUERIES]

# Warm-up
for q in queries[:2]:
    rerank(q, retrieve_top10(q))

lat = []
for q in queries:
    for _ in range(N_RUNS):
        cands, t_ret = _timed(retrieve_top10, q)
        _, t_rr      = _timed(rerank, q, cands)
        lat.append({"query": q, "retrieve_ms": t_ret, "rerank_ms": t_rr})
lat = pd.DataFrame(lat)

# LLM time per call, reused from the generations in section 8 (no extra API calls)
llm_times = [r[k] for r in answer_rows for k in ("t_llm_1_ms", "t_llm_2_ms")]
llm_ms = float(np.mean(llm_times)) if llm_times else float("nan")

summary = pd.DataFrame({
    "stage": ["Bi-encoder retrieval (embed + FAISS top 10)", "Cross-encoder re-rank (10 pairs)", "LLM generation"],
    "mean_ms":   [lat.retrieve_ms.mean(), lat.rerank_ms.mean(), llm_ms],
    "median_ms": [lat.retrieve_ms.median(), lat.rerank_ms.median(), llm_ms],
}).round(2)

p1_total = summary.mean_ms[0] + llm_ms
p2_total = summary.mean_ms[0] + summary.mean_ms[1] + llm_ms
totals = pd.DataFrame({
    "pipeline": ["Pipeline 1 (baseline)", "Pipeline 2 (re-rank)"],
    "retrieval_only_ms": [summary.mean_ms[0], summary.mean_ms[0] + summary.mean_ms[1]],
    "end_to_end_ms":     [p1_total, p2_total],
}).round(2)
totals["rerank_overhead_%_of_retrieval"] = [0.0, round(100 * summary.mean_ms[1] / summary.mean_ms[0], 1)]

summary.to_csv(RESULTS_DIR / "latency_stages.csv", index=False)
totals.to_csv(RESULTS_DIR / "latency_totals.csv", index=False)
print(f"Device: {DEVICE} | LLM: {LLM_NAME} | runs per query: {N_RUNS}")
display(summary); display(totals)

Device: cuda | LLM: gemini-3.7-flash | runs per query: 5


,stage,mean_ms,median_ms
0,Bi-encoder retrieval (embed + FAISS top 10),12.73,11.84
1,Cross-encoder re-rank (10 pairs),82.62,83.18
2,LLM generation,9223.50,9223.50


,pipeline,retrieval_only_ms,end_to_end_ms,rerank_overhead_%_of_retrieval
0,Pipeline 1 (baseline),12.73,9236.23,0.0
1,Pipeline 2 (re-rank),95.35,9318.85,649.0


## 10. Scaling experiment: cross-encoder over the whole corpus

This measures what happens if the cross-encoder scores **every** chunk for one query instead of only 10 candidates. The bi-encoder scales because chunk embeddings are computed once offline, so a query costs one embedding plus a vector search. The cross-encoder has no reusable document representation, so it needs one full transformer forward pass per (query, chunk) pair, for every query.

In [10]:
q = queries[0]
pairs = [(q, t) for t in chunk_texts]

t0 = time.perf_counter()
cross_encoder.predict(pairs, batch_size=64)
t_full_ms = (time.perf_counter() - t0) * 1000
per_pair_ms = t_full_ms / len(pairs)

t_bi_ms = lat.retrieve_ms.mean()
scale = pd.DataFrame({
    "corpus_size_chunks": [len(chunks), 100_000, 1_000_000, 10_000_000],
})
scale["cross_encoder_full_scan_s"] = (scale.corpus_size_chunks * per_pair_ms / 1000).round(2)
scale["two_stage_rerank_10_ms"]    = round(lat.rerank_ms.mean(), 2)
scale.to_csv(RESULTS_DIR / "scaling.csv", index=False)

print(f"Full-corpus CE scan: {len(pairs)} pairs in {t_full_ms:.0f} ms ({per_pair_ms:.3f} ms/pair on {DEVICE})")
print(f"Bi-encoder retrieval over the same corpus: {t_bi_ms:.2f} ms")
print("Extrapolation is linear in corpus size (one forward pass per pair). Exact FAISS search is also linear,")
print("but it is a cheap dot product per chunk, and ANN indexes (HNSW/IVF) make it sub-linear.")
scale

Full-corpus CE scan: 305 pairs in 2654 ms (8.700 ms/pair on cuda)
Bi-encoder retrieval over the same corpus: 12.73 ms
Extrapolation is linear in corpus size (one forward pass per pair). Exact FAISS search is also linear,
but it is a cheap dot product per chunk, and ANN indexes (HNSW/IVF) make it sub-linear.


,corpus_size_chunks,cross_encoder_full_scan_s,two_stage_rerank_10_ms
0,305,2.65,82.62
1,100000,870.01,82.62
2,1000000,8700.12,82.62
3,10000000,87001.24,82.62


## 11. Notes for the report

Use the saved outputs in `results/` (`failure_hunt.csv`, `answers.csv`, `latency_*.csv`, `scaling.csv`) as evidence. The four report sections (architecture comparison, rank-shift and answer analysis, latency table, and scaling) should be written in your own words, based on what you observe above.